# 핀홀 카메라 모델과 Camera Calibration 실습

> **강의자료**: `강의자료/05.01.OpenCV-Calibration.md`

| Part | 주제 |
|------|------|
| Part 1 | 핀홀 카메라 모델 |
| Part 2 | 카메라 캘리브레이션 실전 |
| Part 3 | 퀴즈 & 복습 문제 |

이 실습에서는 핀홀 카메라 수학 모델을 직접 구현하고,
OpenCV를 사용해 체커보드 기반 카메라 캘리브레이션 전체 파이프라인을 실습합니다.

In [ ]:
import numpy as np
import cv2              # pip install opencv-python
import yaml             # pip install pyyaml
import os
import glob
import matplotlib.pyplot as plt

In [ ]:
F_MM       = 4.0     # 렌즈 초점 거리 (mm)
DX         = 0.004   # 픽셀 가로 크기 (mm/pixel)
DY         = 0.004   # 픽셀 세로 크기 (mm/pixel)
IMG_WIDTH  = 640     # 이미지 가로 해상도 (pixel)
IMG_HEIGHT = 480     # 이미지 세로 해상도 (pixel)

---
## Part 1: 핀홀 카메라 모델

### 1.1 3D 세계를 2D 이미지로 바꾸는 원리

- **"빛은 직선으로 나아간다"는 단순한 원리로 3D → 2D 변환**
    - **바늘구멍 사진기 (Pinhole Camera)**: 아주 캄캄한 상자 앞면에 바늘구멍 하나를 뚫음
      물체에서 반사된 빛이 구멍을 통과하여 상자 뒷면(센서)에 맺힘
    - **투영 (Projection)**: 3D 공간의 한 점이 구멍을 통과해 2D 평면의 한 점으로 내려앉는 과정
    - **로봇의 거리 추정**: 로봇은 이 원리를 **거꾸로** 이용
      2D 이미지의 한 점 → 실제 3D 공간의 어떤 선상에서 온 것인지 분석
      2D로 변환되는 순간 **'깊이' 정보가 사라짐** → 수학적 모델로 재계산 필요

### 1.2 Aperture (조리개) — 핀홀의 구멍 크기

- **Aperture : 카메라 렌즈에서 빛이 통과하는 구멍의 크기**

| Aperture 크기 | 빛의 양 | 이미지 선명도 |
|---|---|---|
| 크다 (2mm) | 많음  | 흐릿 |
| 작다 (0.35mm) | 적음  | 선명 |

- **핀홀 카메라 모델의 가정**
    - Aperture = **수학적으로 크기가 0인 점**
    - 모든 빛이 딱 한 점을 통과 → 깔끔한 투영 수식 성립
    - 현실에서는 너무 작으면 **빛 부족 → 어두운 이미지** 문제 발생

### 1.3 렌즈 (Lens) — Aperture 딜레마의 해결책

- **핀홀의 딜레마**
  ```
  구멍 크게 → 빛 많음  but  흐릿
  구멍 작게 → 선명     but  어두움
  ```
- **렌즈의 해결책**: 한 점에서 나온 **여러 줄기 빛**을 모아 film의 **한 점으로 수렴**
  ```
  빨간 점 → 여러 빛줄기 → 렌즈 → film의 한 점
  파란 점 → 여러 빛줄기 → 렌즈 → film의 다른 점
  ```
- **새로운 문제**: 렌즈의 굴절 특성 때문에 → **왜곡(Distortion)** 발생
  → 이를 수학적으로 보정하는 것이 **캘리브레이션**

### 1.4 핵심 개념 — 초점 거리 (Focal Length)

- **초점 거리 $f$ : 카메라의 '확대율'을 결정하는 핵심 값**
    - 카메라의 광학 중심(구멍)과 이미지 평면(센서) 사이의 거리
    - 초점 거리가 **길면** → 물체가 크게 보임 (망원)
    - 초점 거리가 **짧으면** → 넓은 범위를 담음 (광각)

- **렌즈 공식**: $\dfrac{1}{f} = \dfrac{1}{z'} - \dfrac{1}{z}$

- **왜 하나의 $f$가 $f_x, f_y$ 두 개가 되는가?** 픽셀이 정사각형이 아닐 수 있기 때문
  $$f_x = \frac{f}{d_x}, \quad f_y = \frac{f}{d_y}$$

| 기호 | 단위 | 의미 |
|---|---|---|
| $f$ | mm | 광학 중심 → 센서 물리적 거리 |
| $d_x, d_y$ | mm/pixel | 픽셀 하나의 실제 크기 |
| $f_x, f_y$ | pixel | 투영 수식의 스케일 인자 |

In [ ]:
# 초점 거리: 물리적 단위(mm) → 픽셀 단위 변환
fx = F_MM / DX
fy = F_MM / DY

print(f"물리적 초점 거리 f = {F_MM} mm")
print(f"픽셀 크기 dx={DX} mm/px,  dy={DY} mm/px")
print(f"픽셀 단위 초점 거리  fx = {fx:.1f} px,  fy = {fy:.1f} px")
print(f"  → 'fx = {fx:.0f}' 은 초점 거리가 픽셀 {fx:.0f}개 길이에 해당한다는 의미")

### 1.5 핵심 개념 — 주점 (Principal Point)

- **주점 $(c_x, c_y)$ : 카메라 렌즈 중심이 이미지와 만나는 기준점**
    - 카메라 렌즈의 중심(광축)이 이미지 센서와 수직으로 만나는 지점의 픽셀 좌표
    - 이상적인 카메라라면 **이미지의 정중앙**이어야 함
    - 실제 제조 과정의 오차로 **조금씩 치우쳐** 있음
    - 이 값이 틀리면 → 로봇이 물체 위치를 잘못 파악

### 1.6 핵심 개념 — 내부 파라미터 행렬 (Camera Matrix K)

**카메라 행렬 $K$ : 카메라의 고유한 '명함'**

초점 거리 $(f_x, f_y)$ 와 주점 $(c_x, c_y)$ 정보를 하나로 묶은 **3×3 행렬**

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

In [ ]:
# 카메라 행렬 K 직접 구성 (예: 640×480 해상도 카메라)
cx = IMG_WIDTH  / 2  # 주점 x (이상적 중앙)
cy = IMG_HEIGHT / 2  # 주점 y (이상적 중앙)

K = np.array([
    [fx,  0,  cx],
    [ 0, fy,  cy],
    [ 0,  0,   1]
], dtype=np.float64)

print("카메라 행렬 K:")
print(K)
print(f"\nfx={K[0,0]}, fy={K[1,1]}, cx={K[0,2]}, cy={K[1,2]}")

In [ ]:
# 카메라 행렬 K로 3D 점 → 2D 픽셀 좌표 투영 (닮음 삼각형 원리)
# 3D 카메라 좌표 (X, Y, Z): Z는 카메라에서 물체까지의 깊이
X, Y, Z = 0.3, 0.1, 2.0    # 단위: m

# 투영 공식: u = fx * X/Z + cx,  v = fy * Y/Z + cy
u = fx * (X / Z) + cx
v = fy * (Y / Z) + cy

print(f"3D 카메라 좌표: ({X}, {Y}, {Z}) m")
print(f"투영된 픽셀 좌표: u={u:.1f}, v={v:.1f}")
print()

In [ ]:
k1 = np.array([X/Z, Y/Z, 1])
k1

In [ ]:
u1 = K @ k1
u1

In [ ]:
# 역방향: 픽셀 → 정규화 이미지 좌표 (K^-1 적용, 깊이 Z 필요)
x_norm = (u - cx) / fx    # 정규화 x
y_norm = (v - cy) / fy    # 정규화 y
X_back = x_norm * Z
Y_back = y_norm * Z
print(f"역투영 (Z={Z}m 알고 있을 때): X={X_back:.3f}, Y={Y_back:.3f}")

In [ ]:
inv_K = np.linalg.inv(K)
matrix1 = inv_K @ u1
matrix1

### 1.7 실제 카메라 vs 수학 모델

- **현실** : 렌즈 카메라 (밝고 선명)
- **수학 모델** : 핀홀 모델 (수식이 단순하고 깔끔)
- **캘리브레이션이 다리 역할**
  ```
  실제 렌즈 카메라
      ↓
  캘리브레이션으로 왜곡 계수(k1, k2, p1, p2, k3) 측정
      ↓
  왜곡 보정 (Rectification)
      ↓
  핀홀 모델 수식 적용
  ```
- 왜곡을 제거하고 나면 → 렌즈 카메라도 **핀홀 카메라처럼 동작**

### 1.8 렌즈 왜곡의 종류

**방사 왜곡 (Radial Distortion)**
- 렌즈 모양 때문에 이미지 **가장자리로 갈수록 직선이 곡선**으로 변하는 현상
- **배럴 왜곡 (Barrel Distortion)**: 이미지가 **볼록하게 튀어나와** 보임 → 광각 렌즈
- **핀쿠션 왜곡 (Pincushion Distortion)**: 이미지가 **오목하게 들어가** 보임 → 망원 렌즈

**접선 왜곡 (Tangential Distortion)**
- 카메라 제조 시 렌즈와 이미지 센서가 **완벽하게 수평을 이루지 못해** 발생
- 이미지가 비스듬하게 기울어져 보이는 현상
- **해결책**: 카메라 캘리브레이션으로 왜곡 계수 측정 후 **직선화(Rectification)**

### 1.9 좌표계의 종류와 변환 흐름

**픽셀 하나를 보고 실제 위치를 알기까지 4단계 좌표 변환**

| 단계 | 좌표계 | 기준 |
|------|--------|------|
| ① | 3D 월드 좌표계 (World Frame) | 방의 구석 등 환경 기준점 |
| ② | 3D 카메라 좌표계 (Camera Frame) | 카메라 렌즈 중심 |
| ③ | 2D 이미지 좌표계 (Image Plane) | 렌즈를 통해 투영된 가상 2D 평면 |
| ④ | 픽셀 좌표계 (Pixel Frame) | 코드에서 다루는 $(u, v)$ 좌표 |

**로봇의 역방향 계산 흐름:**
$$\text{픽셀 좌표} \xrightarrow{K^{-1}} \text{이미지 좌표} \xrightarrow{\text{깊이}} \text{카메라 좌표} \xrightarrow{[R|t]^{-1}} \text{월드 좌표}$$

**닮음 삼각형 원리 (깊이 추정):**
$$\frac{X_{real}}{Z} = \frac{x_{pixel} \times d_x}{f}$$

**혼란 포인트 정리**
- **혼란 1**: 물리적으로 센서에 상이 거꾸로 맺히지만, 수학 모델에서는 이미지 평면을 렌즈 앞에 놓고 똑바로 선 이미지 사용
- **혼란 2**: $K$의 초점 거리는 **픽셀 단위** ($f_x, f_y$) → 미터 단위로 바꾸려면 비례식 필요
- **혼란 3**: 로봇 본체(X=앞, Z=위) vs 카메라 광학 프레임(Z=앞, X=오른쪽, Y=아래) — 축이 90도 다름

In [ ]:
# 깊이 추정 예시: 알려진 크기의 물체로 거리 계산
X_real_mm = 100.0   # 물체의 실제 가로 크기 (mm), 예: 10cm 마커
x_pixel   = 50.0    # 이미지에서 물체가 차지하는 픽셀 수

# 닮음 삼각형: X_real / Z = (x_pixel * dx) / f  (단위 mm로 통일 → 결과도 mm)
Z_estimated_mm = X_real_mm * F_MM / (x_pixel * DX)
print(f"물체 실제 크기: {X_real_mm:.0f} mm")
print(f"이미지 픽셀 크기: {x_pixel:.0f} px")
print(f"추정된 거리(깊이): Z ≈ {Z_estimated_mm:.1f} mm  ({Z_estimated_mm/1000:.3f} m)")

---
## Part 2: 카메라 캘리브레이션 실전

### 2.1 캘리브레이션이 필요한 이유

**보정되지 않은 카메라는 직선을 곡선으로 인식하거나 물체 위치를 잘못 파악**

- **캘리브레이션 전**: 직선이 휘어짐, 2D → 3D 정확한 변환 불가
- **캘리브레이션 후**: 왜곡 계수로 이미지 **직선화(Rectification)** 가능,
  픽셀 좌표 ↔ 실제 3D 공간 사이의 **정확한 수학적 관계** 정의

### 2.2 체커보드 패턴 사용 이유

- 명암 대비가 명확 → **내부 교차점(Corners)** 을 서브픽셀 단위로 정밀 감지 가능
- 규칙적인 격자 구조 → 실제 3D 좌표를 쉽게 정의 가능
- **주의**: 칸 수가 아닌 **내부 교차점의 개수**!
- $8 \times 6$ 칸 체커보드 → 내부 교차점은 $7 \times 5$개

### 2.3 올바른 체커보드 촬영 방법

- **장수**: 최소 **10~20장** 권장 (복잡한 왜곡 렌즈는 최대 40장)
- **각도**: 체커보드를 **다양한 축 방향**으로 기울여 촬영 (±15°~45° 범위 권장, 다양한 각도가 초점 거리 계산에 필수)
- **거리**: 카메라와 체커보드 사이의 **거리를 변화**시키며 촬영
- **주의사항**: 체커보드는 **평평하고 단단한 판**에 부착, **줌 수준 일정 유지**,
  이미지 **전체 영역**(특히 가장자리)을 골고루 커버

### 2.4 OpenCV 캘리브레이션 코드 흐름

**OpenCV 캘리브레이션 3단계**

1. **`findChessboardCorners()`**: 이미지에서 체커보드 내부 교차점들의 픽셀 좌표 탐색
2. **`cornerSubPix()`**: 감지된 코너 좌표를 **서브픽셀 단위로 정밀화** → 정확도 향상
3. **`calibrateCamera()`**: 2D 이미지 포인트 + 미리 정의된 3D 객체 포인트를 비교하여
   시스템 방정식을 풀어 **카메라 파라미터 계산**

In [ ]:
# 체커보드 설정 (10x7 칸 → 내부 교차점 9x6개)
CHECKERBOARD = (9, 6)       # (가로 교차점 수, 세로 교차점 수)
SQUARE_SIZE  = 0.025        # 체커보드 한 칸의 실제 크기 (m), 예: 2.5cm

# 3D 객체 포인트 준비: (0,0,0), (1,0,0), ..., (8,5,0) 형태로 정의 (Z=0 평면)
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE     # 픽셀 단위가 아닌 실제 크기(m) 적용

print(f"체커보드 설정: {CHECKERBOARD[0]}×{CHECKERBOARD[1]} 내부 교차점")
print(f"총 코너 수: {len(objp)}개")
print(f"첫 번째 코너 3D 좌표: {objp[0]}")
print(f"마지막 코너 3D 좌표: {objp[-1]}")

In [ ]:
# 캘리브레이션 전체 파이프라인
# (실제 이미지 파일이 있어야 실행 가능 — 이미지 경로를 수정하세요)

def run_calibration(image_dir: str, pattern=(9, 6), square_size=0.025):
    """체커보드 이미지들로 카메라 캘리브레이션 수행"""
    objp = np.zeros((pattern[0] * pattern[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:pattern[0], 0:pattern[1]].T.reshape(-1, 2)
    objp *= square_size

    objpoints = []   # 3D 실세계 포인트 목록
    imgpoints = []   # 2D 이미지 포인트 목록
    image_size = None

    # cornerSubPix 종료 기준: 정확도(EPS) 또는 최대 반복 횟수(30회) 중 먼저 만족되면 중단
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    images = glob.glob(os.path.join(image_dir, '*.jpg')) + \
             glob.glob(os.path.join(image_dir, '*.png'))

    image_paths = []  # 코너 검출 성공한 이미지 경로 목록

    print(f"이미지 {len(images)}장 로드")
    for fname in images:
        img  = cv2.imread(fname)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        image_size = gray.shape[::-1]   # (width, height)

        # ① 체커보드 코너 탐색
        ret, corners = cv2.findChessboardCorners(gray, pattern, None)
        if not ret:
            print(f"  코너 미검출: {os.path.basename(fname)}")
            continue

        # ② 서브픽셀 정밀화
        corners_refined = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)

        objpoints.append(objp)
        imgpoints.append(corners_refined)
        image_paths.append(fname)
        print(f"  코너 검출 성공: {os.path.basename(fname)}")

    if len(objpoints) < 3:
        print("이미지 부족 (최소 3장 필요). 캘리브레이션 중단.")
        return None, None, None, None, None, None, None, None

    # ③ 캘리브레이션 계산
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, image_size, None, None
    )
    # ret   : 재투영 오차 RMS (픽셀 단위, 1.0 이하면 양호)
    # mtx   : 카메라 행렬 K (3×3) — fx, fy, cx, cy 포함
    # dist  : 왜곡 계수 (k1, k2, p1, p2, k3) — undistort에 사용
    # rvecs : 각 이미지별 회전 벡터 (체커보드 대비 카메라 자세)
    # tvecs : 각 이미지별 이동 벡터 (체커보드 대비 카메라 위치)
    print(f"\n재투영 오차 (RMS Error): {ret:.4f} px")
    print("카메라 행렬 mtx:\n", mtx)
    print("왜곡 계수 dist:", dist.ravel())
    return mtx, dist, ret, rvecs, tvecs, objpoints, imgpoints, image_paths

In [ ]:
# 실제 이미지가 있는 경우 아래 주석을 해제하고 경로를 수정하세요
mtx, dist, rms, rvecs, tvecs, objpoints, imgpoints, image_paths = run_calibration("/home/jaeholee/work/calib_images")

### 2.5 calibrateCamera() 결과 파라미터

```python
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None
)
```

| 반환값 | 설명 |
|--------|------|
| `ret` | 재투영 오차 (RMS Error) — 작을수록 좋음 |
| `mtx` | 카메라 행렬 $K$ (3×3): $f_x, f_y, c_x, c_y$ 포함 |
| `dist` | 왜곡 계수: 방사 왜곡 $(k_1, k_2, k_3)$ + 접선 왜곡 $(p_1, p_2)$ |
| `rvecs` | 각 이미지의 회전 벡터 (외부 파라미터) |
| `tvecs` | 각 이미지의 이동 벡터 (외부 파라미터) |

### 2.6 품질 평가 — 재투영 오차 (Reprojection Error)

**재투영 오차**: 계산된 카메라 모델로 3D 포인트를 다시 2D에 투영했을 때
실제 감지된 이미지 포인트와의 **거리 차이 (픽셀 단위)**

| RMS Error | 평가 |
|-----------|------|
| 0.5 px 미만 | 산업용 비전 시스템에서 성공적 |
| 1 px 이하  | 대부분의 응용 분야에서 수용 가능 |
| 1 px 초과  | 촬영 데이터 점검 후 재캘리브레이션 필요 |

In [ ]:
# 재투영 오차 계산 함수 (캘리브레이션 후 검증용)
def calc_reprojection_error(objpoints, imgpoints, mtx, dist, rvecs, tvecs, image_paths=None):
    """각 이미지별 재투영 오차를 계산하고 평균과 최대 오차 인덱스를 반환"""
    errors = []
    for i, (op, ip) in enumerate(zip(objpoints, imgpoints)):
        # op(z=0 평면의 3D 점)를 rvecs/tvecs로 카메라 좌표계로 변환 후 2D 이미지 평면에 투영
        proj, _ = cv2.projectPoints(op, rvecs[i], tvecs[i], mtx, dist)
        err = cv2.norm(ip, proj, cv2.NORM_L2) / len(proj)
        errors.append(err)
    mean_err = np.mean(errors)
    worst_idx = int(np.argmax(errors))
    print(f"이미지별 재투영 오차: {[f'{e:.4f}' for e in errors]}")
    print(f"평균 재투영 오차: {mean_err:.4f} px")
    if image_paths:
        print(f"최대 오차 이미지: {os.path.basename(image_paths[worst_idx])} ({errors[worst_idx]:.4f} px)")
    return mean_err, worst_idx

In [ ]:
_, worst_idx = calc_reprojection_error(objpoints, imgpoints, mtx, dist, rvecs, tvecs, image_paths)

### 2.7 undistort()로 왜곡 보정하기

In [ ]:
# 왜곡 보정 예시 (실제 이미지가 있을 때 실행)
def _line_fitting_residual(img, pattern):
    """체커보드 코너를 검출해 각 행/열을 직선 피팅하고 RMSE 잔차 반환"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, pattern, None)
    if not ret:
        return None
    corners = corners.reshape(pattern[1], pattern[0], 2)  # (행, 열, xy)
    residuals = []
    # 각 행: y ~ ax + b 피팅
    for row in corners:
        x, y = row[:, 0], row[:, 1]        # 한 행의 코너 좌표 (수평으로 나열된 9개 점)
        coeffs = np.polyfit(x, y, 1)       # 최소제곱법으로 y = ax + b의 a, b 계산
        y_pred = np.polyval(coeffs, x)     # 피팅된 직선으로 y 예측
        residuals.append(np.sqrt(np.mean((y - y_pred) ** 2)))
    # 각 열: x ~ ay + b 피팅
    for col in corners.T:
        x, y = col[:, 0], col[:, 1]        # 한 열의 코너 좌표 (수직으로 나열된 6개 점)
        coeffs = np.polyfit(y, x, 1)       # 최소제곱법으로 x = ay + b의 a, b 계산
        x_pred = np.polyval(coeffs, y)     # 피팅된 직선으로 x 예측
        residuals.append(np.sqrt(np.mean((x - x_pred) ** 2)))
    return np.mean(residuals)

In [ ]:
def undistort_image(img_path: str, mtx: np.ndarray, dist: np.ndarray, pattern=(9, 6)):
    """캘리브레이션 결과로 이미지 왜곡 보정 및 수치 비교"""
    img = cv2.imread(img_path)
    if img is None:
        print(f"이미지를 불러올 수 없습니다: {img_path}")
        return None

    # 왜곡 보정 적용
    dst = cv2.undistort(img, mtx, dist, None, mtx)

    # 직선 피팅 잔차 비교
    res_before = _line_fitting_residual(img, pattern)
    res_after  = _line_fitting_residual(dst, pattern)
    if res_before and res_after:
        print(f"직선 피팅 잔차 — 보정 전: {res_before:.4f} px  →  보정 후: {res_after:.4f} px  "
              f"(개선율: {(1 - res_after / res_before) * 100:.1f}%)")


    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original (Distorted)")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(dst, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Undistorted")
    axes[1].axis("off")
    plt.suptitle(os.path.basename(img_path), fontsize=11)
    plt.tight_layout()
    plt.show()
    return dst

In [ ]:
# 재투영 오차가 가장 큰 이미지로 왜곡 보정 전후 비교
dst = undistort_image(image_paths[worst_idx], mtx, dist)

### 2.8 결과 저장 및 불러오기

**캘리브레이션 결과는 매번 계산할 필요 없이 파일로 저장하여 재사용**

In [ ]:
# 캘리브레이션 결과 저장 — YAML 형식 (ROS 호환)
def save_calibration_yaml(mtx, dist, filepath="camera_info.yaml"):
    data = {
        'camera_matrix': mtx.tolist(),
        'dist_coeff': dist.tolist()
    }
    with open(filepath, 'w') as f:
        yaml.dump(data, f)
    print(f"YAML 저장 완료: {filepath}")

save_calibration_yaml(mtx, dist)

In [ ]:
# 캘리브레이션 결과 불러오기 — YAML 형식
def load_calibration_yaml(filepath="camera_info.yaml"):
    with open(filepath, 'r') as f:
        data = yaml.safe_load(f)
    mtx  = np.array(data['camera_matrix'], np.float64)
    dist = np.array(data['dist_coeff'], np.float64)
    print(f"YAML 로드 완료: {filepath}")
    print("mtx:\n", mtx)
    print("dist:", dist.ravel())
    return mtx, dist

---
## Part 3: 퀴즈 & 복습 문제

### 연습 문제

In [ ]:
# TODO: 여기에 구현하세요
# 요구사항: 아래 함수를 완성하세요.
#   - 체커보드 이미지 한 장을 받아 코너를 검출하고 시각화합니다.
#   - 반환값: 서브픽셀 정밀화된 코너 좌표 (검출 실패 시 None)
#
# Hint:
#   - cv2.findChessboardCorners(gray, pattern) → ret, corners
#   - cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria) → corners_refined
#   - cv2.drawChessboardCorners(img, pattern, corners_refined, ret) 로 시각화

def detect_corners(img_path: str, pattern=(9, 6)):
    ...

# 완성 후 주석 해제
# corners = detect_corners("calib_images/???.jpg")

In [ ]:
# TODO: 여기에 구현하세요
# 요구사항: 아래 함수를 완성하세요.
#   - 저장된 YAML 파일에서 캘리브레이션 파라미터를 불러와 이미지에 undistort를 적용합니다.
#   - 원본 이미지와 보정된 이미지를 나란히 출력합니다.
#
# Hint:
#   - load_calibration_yaml(yaml_path) 로 mtx, dist 불러오기
#   - cv2.undistort(img, mtx, dist, None, mtx) 로 왜곡 보정
#   - plt.subplots(1, 2) 로 보정 전/후 나란히 표시

def apply_undistort(img_path: str, yaml_path: str):
    ...

# 완성 후 주석 해제
# dst = apply_undistort("calib_images/???.jpg", "camera_info.yaml")

In [ ]:
import cv2
import numpy as np
import glob
import os


# ============================================================
# 1. 사용자 설정 영역
# ============================================================

# OpenCV에서 CHECKERBOARD는 "체크보드 칸 수"가 아니라
# "내부 코너 개수"를 의미한다.
#
# 예:
# 실제 보드가 9칸 x 6칸이면 내부 코너는 8 x 5이다.
#
# 네 코드에서 borderSize = (8, 5)를 썼다면
# 보통 CHECKERBOARD = (8, 5)가 맞을 가능성이 높다.
CHECKERBOARD = (9, 6)

# 체크보드 한 칸의 실제 크기.
#
# 단위는 meter를 추천한다.
# 예:
# 25mm = 0.025
# 30mm = 0.030
#
# 왜 실제 크기를 넣는가?
# - 단순 왜곡 보정만 할 거면 1로 둬도 어느 정도 가능하다.
# - 하지만 tvecs, 즉 카메라와 체크보드 사이 실제 거리/위치를 해석하려면
#   실제 크기를 넣는 것이 맞다.
SQUARE_SIZE = 0.025

# 캘리브레이션용 체크보드 이미지들이 들어 있는 경로.
CALIB_IMAGE_GLOB = "/home/jaeholee/work/calib_images/*.*"

# 원본 캘리브레이션 결과 저장 파일.
#
# 이 파일은 "원본 보정값"을 저장한다.
# 즉, cameraMatrix, distCoeffs, image size 등을 저장한다.
#
# 이 파일은 보정의 기준 데이터이므로 반드시 보관하는 것이 좋다.
CALIB_YAML_PATH = "camera_calibration.yml"

# remap용 map 캐시 저장 파일.
#
# 이 파일은 cameraMatrix/distCoeffs로부터 계산된 map1/map2를 저장한다.
# 라즈베리파이에서 실시간 처리할 때는 이 파일을 바로 읽어서
# cv2.remap()에 넣으면 된다.
#
# 단, 이 파일은 특정 해상도와 alpha 값에 묶인다.
# 예: 640x480, alpha=0.0으로 만든 map이면
#     1280x720 프레임에는 그대로 쓰면 안 된다.
MAP_CACHE_NPZ_PATH = "undistort_maps_640x480_alpha0.npz"

# 테스트 이미지 경로.
TEST_IMAGE_PATH = "./test.jpg"

# 보정 결과 저장 경로.
OUTPUT_IMAGE_PATH = "./test_undistorted.jpg"

# alpha 값은 보정 후 시야 보존 정도를 의미한다.
#
# alpha = 0.0
# - 검은 영역을 최대한 제거한다.
# - 대신 가장자리 일부가 잘릴 수 있다.
#
# alpha = 1.0
# - 원래 시야를 최대한 유지한다.
# - 대신 가장자리에 검은 영역이 생길 수 있다.
#
# 일반적으로 실시간 처리나 마커 인식에서는 0.0 또는 0.3 정도가 무난하다.
ALPHA = 0.0

# 재투영 오차가 너무 큰 이미지를 제거할지 여부.
USE_OUTLIER_FILTER = True


# ============================================================
# 2. 체크보드의 3D 기준 좌표 생성
# ============================================================

def create_object_points(pattern_size, square_size):
    """
    체크보드의 실제 3D 좌표를 만든다.

    카메라 캘리브레이션은 다음 대응 관계를 사용한다.

        실제 세계의 3D 점  <->  이미지 안의 2D 점

    체크보드는 평평한 판이므로 z = 0으로 둔다.

    예:
    CHECKERBOARD = (8, 5), SQUARE_SIZE = 0.025라면

        (0.000, 0.000, 0)
        (0.025, 0.000, 0)
        (0.050, 0.000, 0)
        ...
        (0.000, 0.025, 0)
        (0.025, 0.025, 0)

    이런 식의 실제 좌표가 만들어진다.
    """

    cols, rows = pattern_size

    objp = np.zeros((cols * rows, 3), np.float32)
    objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)

    # 실제 한 칸 크기를 곱해서 물리 단위로 변환한다.
    objp *= square_size

    return objp


# ============================================================
# 3. 체크보드 코너 검출
# ============================================================

def find_chessboard_corners(gray, pattern_size):
    """
    이미지에서 체크보드 내부 코너를 찾는다.

    먼저 findChessboardCornersSB()를 시도한다.
    이 방식은 비교적 최신 방식이고, 기존 findChessboardCorners()보다
    조명/각도 변화에 강한 경우가 있다.

    실패하면 기존 방식인 findChessboardCorners() + cornerSubPix()를 사용한다.

    cornerSubPix()를 쓰는 이유:
    - 기본 코너 검출은 픽셀 단위에 가깝다.
    - 캘리브레이션 품질은 코너 위치 정확도에 크게 영향을 받는다.
    - 그래서 서브픽셀 단위로 코너 위치를 정밀하게 보정한다.
    """

    if hasattr(cv2, "findChessboardCornersSB"):
        sb_flags = cv2.CALIB_CB_NORMALIZE_IMAGE

        ret, corners = cv2.findChessboardCornersSB(
            gray,
            pattern_size,
            flags=sb_flags
        )

        if ret:
            return True, corners.astype(np.float32)

    classic_flags = (
        cv2.CALIB_CB_ADAPTIVE_THRESH |
        cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    ret, corners = cv2.findChessboardCorners(
        gray,
        pattern_size,
        classic_flags
    )

    if not ret:
        return False, None

    criteria = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        40,
        0.001
    )

    corners = cv2.cornerSubPix(
        gray,
        corners,
        winSize=(11, 11),
        zeroZone=(-1, -1),
        criteria=criteria
    )

    return True, corners


# ============================================================
# 4. 캘리브레이션 이미지에서 objpoints / imgpoints 수집
# ============================================================

def collect_calibration_points(image_glob, pattern_size, square_size):
    """
    여러 장의 체크보드 이미지에서 캘리브레이션에 필요한 점들을 수집한다.

    objpoints:
    - 실제 세계의 3D 체크보드 좌표 목록

    imgpoints:
    - 이미지에서 검출된 2D 코너 좌표 목록

    중요한 점:
    - 체크보드 검출에 실패한 이미지는 사용하지 않는다.
    - 이미지 해상도가 서로 다르면 단일 캘리브레이션에 섞지 않는다.
    """

    image_paths = sorted(glob.glob(image_glob))

    if len(image_paths) == 0:
        raise RuntimeError(f"캘리브레이션 이미지를 찾지 못했습니다: {image_glob}")

    base_objp = create_object_points(pattern_size, square_size)

    objpoints = []
    imgpoints = []
    valid_image_paths = []

    image_size = None

    for path in image_paths:
        img = cv2.imread(path)

        if img is None:
            print(f"[SKIP] 이미지 로드 실패: {path}")
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        current_size = gray.shape[::-1]

        if image_size is None:
            image_size = current_size
        elif current_size != image_size:
            print(f"[SKIP] 이미지 크기 다름: {path}, size={current_size}, expected={image_size}")
            continue

        ret, corners = find_chessboard_corners(gray, pattern_size)

        if not ret:
            print(f"[SKIP] 체크보드 코너 검출 실패: {os.path.basename(path)}")
            continue

        objpoints.append(base_objp.copy())
        imgpoints.append(corners)
        valid_image_paths.append(path)

        print(f"[OK] 체크보드 검출 성공: {os.path.basename(path)}")

    if len(objpoints) < 10:
        raise RuntimeError(
            f"검출 성공 이미지가 너무 적습니다: {len(objpoints)}장. "
            f"최소 10장 이상, 가능하면 15~30장 정도를 권장합니다."
        )

    return objpoints, imgpoints, valid_image_paths, image_size


# ============================================================
# 5. 카메라 캘리브레이션 수행
# ============================================================

def calibrate_camera(objpoints, imgpoints, image_size):
    """
    cv2.calibrateCamera()로 카메라 내부 파라미터와 왜곡 계수를 구한다.

    반환값:
    - rms:
      전체 재투영 RMS 오차

    - camera_matrix:
      카메라 내부 파라미터

        [ fx   0  cx ]
        [  0  fy  cy ]
        [  0   0   1 ]

    - dist_coeffs:
      렌즈 왜곡 계수

    - rvecs, tvecs:
      각 체크보드 이미지에서 보드의 회전/이동 정보
    """

    rms, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        objpoints,
        imgpoints,
        image_size,
        None,
        None
    )

    return rms, camera_matrix, dist_coeffs, rvecs, tvecs


# ============================================================
# 6. 재투영 오차 계산
# ============================================================

def compute_reprojection_errors(objpoints, imgpoints, rvecs, tvecs, camera_matrix, dist_coeffs):
    """
    이미지별 재투영 오차를 계산한다.

    재투영 오차란:
    - 실제 3D 체크보드 점을 cameraMatrix/distCoeffs를 사용해 다시 이미지에 투영한다.
    - 그 투영된 점과 실제 검출된 2D 코너 사이의 차이를 계산한다.

    이 값이 큰 이미지는 다음 가능성이 있다.
    - 체크보드가 흐릿함
    - 코너 검출이 이상함
    - 빛 반사가 있음
    - 보드가 너무 작거나 너무 비스듬함
    """

    errors = []

    for i in range(len(objpoints)):
        projected_points, _ = cv2.projectPoints(
            objpoints[i],
            rvecs[i],
            tvecs[i],
            camera_matrix,
            dist_coeffs
        )

        error = cv2.norm(
            imgpoints[i],
            projected_points,
            cv2.NORM_L2
        ) / len(projected_points)

        errors.append(error)

    return np.array(errors, dtype=np.float64)


# ============================================================
# 7. 오차 큰 이미지 제거 후 재캘리브레이션
# ============================================================

def filter_outliers_and_recalibrate(objpoints, imgpoints, image_paths, image_size):
    """
    1차 캘리브레이션을 수행한 뒤,
    이미지별 재투영 오차가 유난히 큰 이미지를 제거하고 다시 캘리브레이션한다.

    왜 하는가?
    - 캘리브레이션은 이미지 몇 장이 이상해도 전체 결과가 흔들릴 수 있다.
    - 따라서 오차가 유난히 큰 이미지들을 제거하면 결과가 더 안정될 수 있다.

    단, 이미지를 너무 많이 제거하면 다양한 각도 정보가 사라진다.
    그래서 극단적인 outlier만 제거하는 것이 좋다.
    """

    rms, camera_matrix, dist_coeffs, rvecs, tvecs = calibrate_camera(
        objpoints,
        imgpoints,
        image_size
    )

    errors = compute_reprojection_errors(
        objpoints,
        imgpoints,
        rvecs,
        tvecs,
        camera_matrix,
        dist_coeffs
    )

    mean_error = float(np.mean(errors))
    std_error = float(np.std(errors))
    threshold = mean_error + 2.0 * std_error

    print()
    print("========== 1차 캘리브레이션 결과 ==========")
    print(f"RMS error         : {rms}")
    print(f"Mean reproj error : {mean_error}")
    print(f"Std reproj error  : {std_error}")
    print(f"Outlier threshold : {threshold}")
    print()

    filtered_objpoints = []
    filtered_imgpoints = []
    filtered_image_paths = []

    for i, err in enumerate(errors):
        name = os.path.basename(image_paths[i])

        if err <= threshold:
            print(f"[KEEP] {name}, reproj_error={err:.6f}")
            filtered_objpoints.append(objpoints[i])
            filtered_imgpoints.append(imgpoints[i])
            filtered_image_paths.append(image_paths[i])
        else:
            print(f"[DROP] {name}, reproj_error={err:.6f}")

    if len(filtered_objpoints) < 10:
        print()
        print("[WARN] outlier 제거 후 이미지가 너무 적습니다.")
        print("[WARN] 1차 캘리브레이션 결과를 그대로 사용합니다.")
        return rms, camera_matrix, dist_coeffs, rvecs, tvecs, errors, image_paths

    rms2, camera_matrix2, dist_coeffs2, rvecs2, tvecs2 = calibrate_camera(
        filtered_objpoints,
        filtered_imgpoints,
        image_size
    )

    errors2 = compute_reprojection_errors(
        filtered_objpoints,
        filtered_imgpoints,
        rvecs2,
        tvecs2,
        camera_matrix2,
        dist_coeffs2
    )

    print()
    print("========== 2차 캘리브레이션 결과 ==========")
    print(f"RMS error         : {rms2}")
    print(f"Mean reproj error : {float(np.mean(errors2))}")
    print(f"Std reproj error  : {float(np.std(errors2))}")
    print(f"Used images       : {len(filtered_objpoints)}")
    print()

    return rms2, camera_matrix2, dist_coeffs2, rvecs2, tvecs2, errors2, filtered_image_paths


# ============================================================
# 8. YAML 저장 / 로드
# ============================================================

def save_calibration_to_yaml(path, camera_matrix, dist_coeffs, image_size, square_size, pattern_size, rms):
    """
    원본 캘리브레이션 결과를 YAML로 저장한다.

    이 파일에는 다음이 들어간다.
    - image_width
    - image_height
    - checkerboard_cols
    - checkerboard_rows
    - square_size
    - rms
    - camera_matrix
    - dist_coeffs

    왜 YAML을 저장해야 하는가?
    - 이 파일이 진짜 원본 보정값이다.
    - 사람이 열어서 확인할 수 있다.
    - 나중에 alpha 값을 바꾸거나, map을 다시 만들 수 있다.
    - C++ OpenCV, ROS 쪽과 연결하기도 상대적으로 좋다.

    즉:
    YAML = 원본 보정 데이터
    NPZ  = 실시간 remap용 캐시
    """

    fs = cv2.FileStorage(path, cv2.FILE_STORAGE_WRITE)

    if not fs.isOpened():
        raise RuntimeError(f"YAML 파일을 열 수 없습니다: {path}")

    fs.write("image_width", int(image_size[0]))
    fs.write("image_height", int(image_size[1]))
    fs.write("checkerboard_cols", int(pattern_size[0]))
    fs.write("checkerboard_rows", int(pattern_size[1]))
    fs.write("square_size", float(square_size))
    fs.write("rms", float(rms))
    fs.write("camera_matrix", camera_matrix)
    fs.write("dist_coeffs", dist_coeffs)

    fs.release()

    print(f"[SAVE] YAML 저장 완료: {path}")


def load_calibration_from_yaml(path):
    """
    YAML에서 원본 캘리브레이션 값을 읽는다.

    이 함수는 라즈베리파이에서도 그대로 쓸 수 있다.
    YAML만 있으면 map1/map2를 다시 생성할 수 있다.
    """

    fs = cv2.FileStorage(path, cv2.FILE_STORAGE_READ)

    if not fs.isOpened():
        raise RuntimeError(f"YAML 파일을 열 수 없습니다: {path}")

    image_width = int(fs.getNode("image_width").real())
    image_height = int(fs.getNode("image_height").real())
    camera_matrix = fs.getNode("camera_matrix").mat()
    dist_coeffs = fs.getNode("dist_coeffs").mat()

    fs.release()

    image_size = (image_width, image_height)

    return camera_matrix, dist_coeffs, image_size


# ============================================================
# 9. remap용 map 생성
# ============================================================

def create_undistort_maps(camera_matrix, dist_coeffs, image_size, alpha):
    """
    cameraMatrix와 distCoeffs를 이용해서 remap용 map1, map2를 만든다.

    cv2.undistort()와 remap 방식의 차이:

    cv2.undistort()
    - 한 장 테스트하기에는 간단하다.
    - 하지만 호출할 때마다 내부적으로 보정 좌표 계산이 들어간다.

    initUndistortRectifyMap() + remap()
    - map1/map2를 미리 만든다.
    - 이후 매 프레임 remap만 수행한다.
    - 라즈베리파이 실시간 처리에 더 적합하다.
    """

    new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix,
        dist_coeffs,
        image_size,
        alpha
    )

    map1, map2 = cv2.initUndistortRectifyMap(
        camera_matrix,
        dist_coeffs,
        None,
        new_camera_matrix,
        image_size,
        cv2.CV_32F
    )

    return new_camera_matrix, roi, map1, map2


# ============================================================
# 10. NPZ 방식 map 캐시 저장 / 로드
# ============================================================

def save_undistort_maps_to_npz(path, map1, map2, image_size, alpha, new_camera_matrix, roi):
    """
    라즈베리파이 실시간 실행용 map 캐시를 NPZ로 저장한다.

    이 파일은 원본 보정값이 아니라,
    remap에 바로 넣을 수 있는 map1/map2를 저장한다.

    장점:
    - 라즈베리파이 실행 시 map 생성 과정을 생략할 수 있다.
    - np.load() 후 바로 cv2.remap()에 사용할 수 있다.
    - map1, map2를 이름으로 저장하므로 dat 방식보다 안전하다.

    단점:
    - 특정 해상도에 묶인다.
    - 특정 alpha 값에 묶인다.
    - 특정 newCameraMatrix에 묶인다.
    - 카메라 해상도나 alpha 정책이 바뀌면 다시 만들어야 한다.

    따라서 이 파일은 "원본"이 아니라 "캐시"로 봐야 한다.
    """

    np.savez(
        path,
        map1=map1,
        map2=map2,
        image_width=int(image_size[0]),
        image_height=int(image_size[1]),
        alpha=float(alpha),
        new_camera_matrix=new_camera_matrix,
        roi=np.array(roi, dtype=np.int32)
    )

    print(f"[SAVE] remap 캐시 NPZ 저장 완료: {path}")


def load_undistort_maps_from_npz(path):
    """
    NPZ에서 map1/map2를 읽는다.

    라즈베리파이 실시간 코드에서는 이 함수로 map을 읽고,
    매 프레임 다음처럼 적용하면 된다.

        undistorted = cv2.remap(frame, map1, map2, cv2.INTER_LINEAR)

    """

    data = np.load(path)

    map1 = data["map1"]
    map2 = data["map2"]

    image_width = int(data["image_width"])
    image_height = int(data["image_height"])

    alpha = float(data["alpha"])

    new_camera_matrix = data["new_camera_matrix"]
    
    roi = tuple(data["roi"].tolist())

    image_size = (image_width, image_height)

    return map1, map2, image_size, alpha, new_camera_matrix, roi


# ============================================================
# 11. 다른 곳에서 쓰는 dat 방식 참고
# ============================================================

def save_undistort_maps_dat_style_example_comment_only():
    """
    다른 코드나 예제에서 아래 같은 방식을 볼 수 있다.

        f = open('calib.dat', 'wb')
        np.save(f, map1)
        np.save(f, map2)
        f.close()

    읽을 때는 순서대로 읽어야 한다.

        f = open('calib.dat', 'rb')
        map1 = np.load(f)
        map2 = np.load(f)
        f.close()

    이 방식도 동작은 한다.

    하지만 단점이 있다.
    - map1, map2라는 이름이 파일 안에 명확히 남지 않는다.
    - 저장 순서를 기억해야 한다.
    - 중간에 다른 데이터를 추가하면 읽는 순서가 꼬일 수 있다.
    - image_width, image_height, alpha 같은 메타데이터를 같이 관리하기 불편하다.

    그래서 Python 프로젝트에서는 np.savez() 방식이 더 안전하다.

    정리:
    - calib.dat에 np.save 두 번 하는 방식:
      가능은 하지만 순서 의존적이다.

    - np.savez() 방식:
      map1, map2, image_width, image_height, alpha 등을 이름으로 저장하므로
      관리가 더 쉽고 실수가 적다.

    따라서 이 코드에서는 dat 방식 대신 NPZ 방식을 사용한다.
    """
    pass


# ============================================================
# 12. 이미지 보정
# ============================================================

def undistort_image_with_remap(image, map1, map2, roi=None):
    """
    map1/map2를 이용해 이미지의 렌즈 왜곡을 보정한다.

    remap은 출력 이미지의 각 픽셀이 원본 이미지의 어느 좌표에서 값을 가져올지
    map1/map2를 보고 계산한다.

    INTER_LINEAR:
    - 보간 방식이다.
    - 원본 좌표가 정수 픽셀에 딱 맞지 않을 때 주변 픽셀을 섞어서 값을 만든다.
    """

    undistorted = cv2.remap(
        image,
        map1,
        map2,
        interpolation=cv2.INTER_LINEAR
    )

    if roi is not None:
        x, y, w, h = roi

        if w > 0 and h > 0:
            undistorted = undistorted[y:y + h, x:x + w]

    return undistorted


# ============================================================
# 13. 전체 캘리브레이션 실행
# ============================================================

def run_calibration():
    """
    체크보드 이미지 여러 장을 이용해 캘리브레이션을 수행한다.

    최종적으로 두 개의 파일을 저장한다.

    1. camera_calibration.yml
       - 원본 보정값
       - cameraMatrix, distCoeffs 저장
       - 반드시 보관 추천

    2. undistort_maps_640x480_alpha0.npz
       - remap용 map1/map2 캐시
       - 라즈베리파이 실시간 실행용
       - 같은 해상도/같은 alpha일 때 바로 사용 가능
    """

    objpoints, imgpoints, valid_image_paths, image_size = collect_calibration_points(
        CALIB_IMAGE_GLOB,
        CHECKERBOARD,
        SQUARE_SIZE
    )

    if USE_OUTLIER_FILTER:
        rms, camera_matrix, dist_coeffs, rvecs, tvecs, errors, used_image_paths = filter_outliers_and_recalibrate(
            objpoints,
            imgpoints,
            valid_image_paths,
            image_size
        )
    else:
        rms, camera_matrix, dist_coeffs, rvecs, tvecs = calibrate_camera(
            objpoints,
            imgpoints,
            image_size
        )

        errors = compute_reprojection_errors(
            objpoints,
            imgpoints,
            rvecs,
            tvecs,
            camera_matrix,
            dist_coeffs
        )

        used_image_paths = valid_image_paths

    print()
    print("========== 최종 캘리브레이션 결과 ==========")
    print(f"Image size      : {image_size}")
    print(f"Used images     : {len(used_image_paths)}")
    print(f"RMS error       : {rms}")
    print(f"Mean reproj err : {float(np.mean(errors))}")
    print("Camera matrix:")
    print(camera_matrix)
    print("Dist coeffs:")
    print(dist_coeffs.ravel())
    print()

    # ------------------------------------------------------------
    # 1. 원본 캘리브레이션 결과 저장
    # ------------------------------------------------------------
    #
    # 이 파일은 반드시 저장하는 것을 추천한다.
    # 나중에 map1/map2를 다시 만들거나 alpha 값을 바꿀 수 있기 때문이다.
    save_calibration_to_yaml(
        CALIB_YAML_PATH,
        camera_matrix,
        dist_coeffs,
        image_size,
        SQUARE_SIZE,
        CHECKERBOARD,
        rms
    )

    # ------------------------------------------------------------
    # 2. 라즈베리파이 실시간 실행용 map 캐시 생성
    # ------------------------------------------------------------
    #
    # cameraMatrix/distCoeffs로부터 map1/map2를 만든다.
    # 이 map은 특정 image_size와 ALPHA에 묶인다.
    new_camera_matrix, roi, map1, map2 = create_undistort_maps(
        camera_matrix,
        dist_coeffs,
        image_size,
        ALPHA
    )

    # ------------------------------------------------------------
    # 3. NPZ 방식으로 map 캐시 저장
    # ------------------------------------------------------------
    #
    # 이 파일을 라즈베리파이에 복사하면
    # 라즈베리파이 실행 코드에서 바로 np.load()로 읽고
    # cv2.remap()에 사용할 수 있다.
    save_undistort_maps_to_npz(
        MAP_CACHE_NPZ_PATH,
        map1,
        map2,
        image_size,
        ALPHA,
        new_camera_matrix,
        roi
    )


# ============================================================
# 14. YAML에서 map 생성 후 테스트 이미지 보정
# ============================================================

def run_undistort_test_from_yaml():
    """
    YAML을 읽고 map1/map2를 새로 생성한 뒤 테스트 이미지를 보정한다.

    이 방식은 원본 캘리브레이션 값을 기준으로 매번 map을 새로 만든다.

    장점:
    - alpha 값을 바꿔서 테스트하기 쉽다.
    - YAML만 있으면 언제든 map을 재생성할 수 있다.

    단점:
    - 실행할 때마다 initUndistortRectifyMap()을 수행한다.
    - 그래도 보통 시작 시 한 번만 수행하므로 큰 문제는 아니다.
    """

    camera_matrix, dist_coeffs, calib_image_size = load_calibration_from_yaml(
        CALIB_YAML_PATH
    )

    image = cv2.imread(TEST_IMAGE_PATH)

    if image is None:
        raise RuntimeError(f"테스트 이미지를 읽을 수 없습니다: {TEST_IMAGE_PATH}")

    current_image_size = (image.shape[1], image.shape[0])

    if current_image_size != calib_image_size:
        raise RuntimeError(
            f"테스트 이미지 해상도가 캘리브레이션 해상도와 다릅니다. "
            f"test={current_image_size}, calib={calib_image_size}"
        )

    new_camera_matrix, roi, map1, map2 = create_undistort_maps(
        camera_matrix,
        dist_coeffs,
        calib_image_size,
        ALPHA
    )

    undistorted = undistort_image_with_remap(
        image,
        map1,
        map2,
        roi=None
    )

    cv2.imwrite(OUTPUT_IMAGE_PATH, undistorted)

    print(f"[SAVE] YAML 기반 보정 이미지 저장 완료: {OUTPUT_IMAGE_PATH}")


# ============================================================
# 15. NPZ map 캐시에서 바로 테스트 이미지 보정
# ============================================================

def run_undistort_test_from_npz():
    """
    NPZ에 저장된 map1/map2를 바로 읽어서 테스트 이미지를 보정한다.

    이 방식이 라즈베리파이 실시간 실행에 가깝다.

    흐름:
    - np.load()로 map1/map2 읽기
    - 카메라 프레임 또는 이미지에 cv2.remap() 적용

    장점:
    - 실행 시 cameraMatrix -> map1/map2 생성 과정을 생략할 수 있다.
    - 실시간 처리 구조가 단순하다.

    단점:
    - 해상도와 alpha가 고정된다.
    - 조건이 바뀌면 NPZ를 다시 만들어야 한다.
    """

    map1, map2, map_image_size, alpha, new_camera_matrix, roi = load_undistort_maps_from_npz(
        MAP_CACHE_NPZ_PATH
    )

    image = cv2.imread(TEST_IMAGE_PATH)

    if image is None:
        raise RuntimeError(f"테스트 이미지를 읽을 수 없습니다: {TEST_IMAGE_PATH}")

    current_image_size = (image.shape[1], image.shape[0])

    if current_image_size != map_image_size:
        raise RuntimeError(
            f"테스트 이미지 해상도가 map 캐시 해상도와 다릅니다. "
            f"test={current_image_size}, map={map_image_size}"
        )

    undistorted = undistort_image_with_remap(
        image,
        map1,
        map2,
        roi=None
    )

    cv2.imwrite(OUTPUT_IMAGE_PATH, undistorted)

    print(f"[SAVE] NPZ map 캐시 기반 보정 이미지 저장 완료: {OUTPUT_IMAGE_PATH}")


# ============================================================
# 16. main
# ============================================================

if __name__ == "__main__":
    """
    사용 순서:

    1. 체크보드 이미지들을 ./calib_images/ 폴더에 넣는다.

        ./calib_images/calib_001.jpg
        ./calib_images/calib_002.jpg
        ./calib_images/calib_003.jpg
        ...

    2. CHECKERBOARD 값을 실제 내부 코너 개수에 맞춘다.

        내부 코너가 8 x 5이면:
        CHECKERBOARD = (8, 5)

        내부 코너가 9 x 6이면:
        CHECKERBOARD = (9, 6)

    3. SQUARE_SIZE를 실제 한 칸 크기로 맞춘다.

        한 칸이 25mm이면:
        SQUARE_SIZE = 0.025

    4. run_calibration() 실행

        생성되는 파일:
        - camera_calibration.yml
        - undistort_maps_640x480_alpha0.npz

    5. 라즈베리파이에 옮길 때는 둘 다 옮기는 것을 추천한다.

        camera_calibration.yml
        - 원본 보정값
        - 나중에 map 재생성 가능

        undistort_maps_640x480_alpha0.npz
        - 실시간 remap용 캐시
        - 바로 cv2.remap()에 사용 가능

    6. 라즈베리파이 실시간 코드에서는 보통 NPZ를 읽어서 쓴다.

        data = np.load("undistort_maps_640x480_alpha0.npz")
        map1 = data["map1"]
        map2 = data["map2"]

        frame_undistorted = cv2.remap(frame, map1, map2, cv2.INTER_LINEAR)

    7. 단, 카메라 해상도나 alpha 값을 바꾸면
       NPZ map 캐시는 다시 만들어야 한다.
    """

    # 1단계:
    # 체크보드 이미지들로 캘리브레이션 수행
    # YAML 원본 보정값 저장
    # NPZ remap 캐시 저장
    run_calibration()

    # 2단계:
    # YAML에서 읽어서 테스트 이미지 보정
    # TEST_IMAGE_PATH가 준비되어 있을 때만 사용한다.
    # run_undistort_test_from_yaml()

    # 3단계:
    # NPZ map 캐시에서 바로 읽어서 테스트 이미지 보정
    # 라즈베리파이 실시간 실행 구조와 가장 비슷하다.
    # TEST_IMAGE_PATH가 준비되어 있을 때만 사용한다.
    # run_undistort_test_from_npz()

[OK] 체크보드 검출 성공: calib_20260429_114148_723839.jpg
[OK] 체크보드 검출 성공: calib_20260429_114151_572890.jpg
[OK] 체크보드 검출 성공: calib_20260429_114154_240935.jpg
[OK] 체크보드 검출 성공: calib_20260429_114157_500385.jpg
[OK] 체크보드 검출 성공: calib_20260429_114200_695924.jpg
[OK] 체크보드 검출 성공: calib_20260429_114202_932765.jpg
[OK] 체크보드 검출 성공: calib_20260429_114208_759279.jpg
[OK] 체크보드 검출 성공: calib_20260429_114214_411672.jpg
[OK] 체크보드 검출 성공: calib_20260429_114215_480238.jpg
[OK] 체크보드 검출 성공: calib_20260429_114218_640864.jpg
[OK] 체크보드 검출 성공: calib_20260429_114221_617284.jpg
[OK] 체크보드 검출 성공: calib_20260429_114226_642275.jpg
[OK] 체크보드 검출 성공: calib_20260429_114235_720648.jpg
[OK] 체크보드 검출 성공: calib_20260429_114239_519916.jpg
[OK] 체크보드 검출 성공: calib_20260429_114240_211544.jpg
[OK] 체크보드 검출 성공: calib_20260429_114244_753742.jpg

========== 1차 캘리브레이션 결과 ==========
RMS error         : 0.5157862285898495
Mean reproj error : 0.06463043640727667
Std reproj error  : 0.02737671828722044
Outlier threshold : 0.11938387298171754

[KEE